In [ ]:
# %% [markdown]
# # Chile Supply Chain: Data Quality and Diagnostic Visualizations
#
# Non-network visualizations for the Chile mineral pipeline.
# Run after `Chile_Pipeline.py` (requires inventory CSV in `Preliminary/`).
#
# **Outputs:** `Outputs/Missing_Data_Analysis.png`

# %% 0. Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os

plt.rcParams.update({
    "font.family": "sans-serif", "font.size": 8.5,
    "axes.titlesize": 10, "axes.labelsize": 9,
    "figure.facecolor": "white", "axes.facecolor": "#fafafa",
    "axes.edgecolor": "#cccccc", "grid.color": "#e0e0e0", "grid.linewidth": 0.4,
})

BASE_DIR = "/Users/leoss/Desktop/GitHub/Capstone/Case studies/Chile"
PRELIM = os.path.join(BASE_DIR, "Preliminary")
OUT_DIR = os.path.join(BASE_DIR, "Outputs")

FACILITY_STAGE = {
    "Mine (active)": "extraction", "Mine (idle)": "extraction",
    "Prospect/Project": "extraction", "Mine (USGS)": "extraction",
    "Concentrator": "processing", "SX-EW Plant": "processing",
    "Smelter": "processing", "Refinery": "processing",
    "Processing Plant": "processing", "Pellet Plant": "processing",
    "Grinding Plant": "processing", "Steel Plant": "processing",
}

inv = pd.read_csv(os.path.join(PRELIM, "Chile_Minerals_Inventory.csv"))

# %% 1. Missing resource/reserve data analysis
res_cols = sorted([c for c in inv.columns if c.endswith(("_Resource", "_Reserve"))])
commodities_with_data = sorted(set(c.rsplit("_", 1)[0] for c in res_cols))

EXTRACTION_TYPES = ["Mine (active)", "Mine (idle)", "Prospect/Project", "Mine (USGS)"]
mines = inv[inv["FACILITY_TYPE"].isin(EXTRACTION_TYPES)].copy()
mines["has_any_resource"] = mines[res_cols].notna().any(axis=1)

def parse_comms(val):
    if pd.isna(val):
        return []
    return [x.strip() for x in str(val).replace(",", "-").replace("/", "-").split("-") if x.strip()]

if "ALL_COMMODITIES_RAW" in mines.columns:
    mines["_comm_list"] = mines["ALL_COMMODITIES_RAW"].apply(parse_comms)
elif "COMMODITY_LIST_STR" in mines.columns:
    mines["_comm_list"] = mines["COMMODITY_LIST_STR"].apply(
        lambda x: [c.strip() for c in str(x).split(",")] if pd.notna(x) else [])
else:
    mines["_comm_list"] = mines["PRIMARY_COMMODITY"].apply(lambda x: [x] if pd.notna(x) else [])

def has_primary_resource(row):
    pc = row.get("PRIMARY_COMMODITY", "")
    if pd.isna(pc) or pc == "":
        return False
    res_col = f"{pc}_Resource"
    rev_col = f"{pc}_Reserve"
    has_res = res_col in inv.columns and pd.notna(row.get(res_col))
    has_rev = rev_col in inv.columns and pd.notna(row.get(rev_col))
    return has_res or has_rev

mines["has_primary_resource"] = mines.apply(has_primary_resource, axis=1)

COL_HAS = "#2d6a4f"
COL_MISS = "#c1121f"

fig = plt.figure(figsize=(16, 13))
fig.suptitle("Resource / Reserve Data Coverage: Chile Mineral Deposits",
             fontsize=13, fontweight="bold", y=0.98)
fig.text(0.5, 0.955,
         f"{len(mines)} extraction-stage facilities | {len(res_cols)} resource/reserve fields | "
         f"{len(commodities_with_data)} commodities with data",
         ha="center", fontsize=9, color="#555555")

gs = gridspec.GridSpec(3, 2, hspace=0.38, wspace=0.30,
                       left=0.07, right=0.95, top=0.93, bottom=0.05)

# Panel 0: Global summary
ax0 = fig.add_subplot(gs[0, 0])
global_stats = {
    "Has resource OR\nreserve data": mines["has_any_resource"].sum(),
    "No resource/\nreserve data": (~mines["has_any_resource"]).sum(),
}
bars = ax0.barh(list(global_stats.keys()), list(global_stats.values()),
                color=[COL_HAS, COL_MISS], edgecolor="white", height=0.5)
for bar, val in zip(bars, global_stats.values()):
    pct = val / len(mines) * 100
    ax0.text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2,
             f"{val}  ({pct:.1f}%)", va="center", fontsize=9)
ax0.set_xlim(0, max(global_stats.values()) * 1.35)
ax0.set_title("Global Coverage: Any Resource/Reserve Data", fontweight="bold")
ax0.set_xlabel("Number of deposits")
ax0.invert_yaxis(); ax0.grid(axis="x", alpha=0.3)

# Panel 1: Coverage by facility type
ax1 = fig.add_subplot(gs[0, 1])
type_stats = mines.groupby("FACILITY_TYPE").agg(
    total=("has_any_resource", "size"),
    has_data=("has_any_resource", "sum")
).sort_values("total", ascending=True)
type_stats["missing"] = type_stats["total"] - type_stats["has_data"]
type_stats["pct_has"] = (type_stats["has_data"] / type_stats["total"] * 100).round(1)

y_pos = range(len(type_stats))
ax1.barh(y_pos, type_stats["has_data"], color=COL_HAS, label="Has data", height=0.6)
ax1.barh(y_pos, type_stats["missing"], left=type_stats["has_data"],
         color=COL_MISS, label="Missing", height=0.6, alpha=0.7)
ax1.set_yticks(y_pos); ax1.set_yticklabels(type_stats.index)
for i, (_, row) in enumerate(type_stats.iterrows()):
    ax1.text(row["total"] + 1, i, f'{row["pct_has"]:.0f}% covered  (n={row["total"]})',
             va="center", fontsize=8, color="#333")
ax1.set_xlim(0, type_stats["total"].max() * 1.45)
ax1.set_title("Coverage by Facility Type", fontweight="bold")
ax1.legend(loc="lower right", fontsize=8, framealpha=0.9); ax1.grid(axis="x", alpha=0.3)

# Panel 2: Coverage by commodity
ax2 = fig.add_subplot(gs[1, 0])
all_comms = mines["PRIMARY_COMMODITY"].dropna().unique()
comm_coverage = []
for comm in sorted(all_comms):
    subset = mines[mines["PRIMARY_COMMODITY"] == comm]
    total = len(subset)
    res_col_name = f"{comm}_Resource"
    rev_col_name = f"{comm}_Reserve"
    n_either = subset.apply(
        lambda r: pd.notna(r.get(res_col_name)) or pd.notna(r.get(rev_col_name)), axis=1).sum() \
        if (res_col_name in subset.columns or rev_col_name in subset.columns) else 0
    n_res = subset[res_col_name].notna().sum() if res_col_name in subset.columns else 0
    n_rev = subset[rev_col_name].notna().sum() if rev_col_name in subset.columns else 0
    comm_coverage.append({
        "commodity": comm, "total": total,
        "has_resource": int(n_res), "has_reserve": int(n_rev),
        "has_either": int(n_either), "missing": total - int(n_either),
        "pct": round(n_either / total * 100, 1) if total > 0 else 0,
    })

cc = pd.DataFrame(comm_coverage).sort_values("total", ascending=True)
cc = cc[cc["total"] >= 1]
y_pos = range(len(cc))
ax2.barh(y_pos, cc["has_either"], color=COL_HAS, label="Has data", height=0.7)
ax2.barh(y_pos, cc["missing"], left=cc["has_either"],
         color=COL_MISS, label="Missing", height=0.7, alpha=0.7)
ax2.set_yticks(y_pos); ax2.set_yticklabels(cc["commodity"], fontsize=8)
for i, (_, row) in enumerate(cc.iterrows()):
    ax2.text(row["total"] + 0.5, i, f'{row["pct"]:.0f}%  ({row["has_either"]}/{row["total"]})',
             va="center", fontsize=7.5, color="#333")
ax2.set_xlim(0, cc["total"].max() * 1.4)
ax2.set_title("Coverage by Primary Commodity", fontweight="bold")
ax2.set_xlabel("Number of deposits")
ax2.legend(loc="lower right", fontsize=8, framealpha=0.9); ax2.grid(axis="x", alpha=0.3)

# Panel 3: Coverage by region
ax3 = fig.add_subplot(gs[1, 1])
reg_col = None
for candidate in ["REGION", "NOMBRE_REGION", "ADM1"]:
    if candidate in mines.columns and mines[candidate].notna().sum() > 10:
        reg_col = candidate
        break

if reg_col:
    reg_stats = mines.groupby(reg_col).agg(
        total=("has_any_resource", "size"),
        has_data=("has_any_resource", "sum")
    ).sort_values("total", ascending=True)
    reg_stats["missing"] = reg_stats["total"] - reg_stats["has_data"]
    reg_stats["pct"] = (reg_stats["has_data"] / reg_stats["total"] * 100).round(1)

    labels = [str(r)[:30] for r in reg_stats.index]
    y_pos = range(len(reg_stats))
    ax3.barh(y_pos, reg_stats["has_data"], color=COL_HAS, label="Has data", height=0.7)
    ax3.barh(y_pos, reg_stats["missing"], left=reg_stats["has_data"],
             color=COL_MISS, label="Missing", height=0.7, alpha=0.7)
    ax3.set_yticks(y_pos); ax3.set_yticklabels(labels, fontsize=7.5)
    for i, (_, row) in enumerate(reg_stats.iterrows()):
        ax3.text(row["total"] + 0.3, i,
                 f'{row["pct"]:.0f}%  ({row["has_data"]}/{row["total"]})',
                 va="center", fontsize=7.5, color="#333")
    ax3.set_xlim(0, reg_stats["total"].max() * 1.4)
    ax3.set_title(f"Coverage by Region ({reg_col})", fontweight="bold")
    ax3.legend(loc="lower right", fontsize=8, framealpha=0.9)
else:
    ax3.text(0.5, 0.5, "No region column found", ha="center", va="center", transform=ax3.transAxes)
    ax3.set_title("Coverage by Region", fontweight="bold")
ax3.grid(axis="x", alpha=0.3)

# Panel 4: Heatmap - commodity x resource/reserve fill
ax4 = fig.add_subplot(gs[2, :])
heat_data = []
for comm in sorted(all_comms):
    subset = mines[mines["PRIMARY_COMMODITY"] == comm]
    total = len(subset)
    if total == 0:
        continue
    res_col_name = f"{comm}_Resource"
    rev_col_name = f"{comm}_Reserve"
    n_res = subset[res_col_name].notna().sum() if res_col_name in subset.columns else 0
    n_rev = subset[rev_col_name].notna().sum() if rev_col_name in subset.columns else 0
    heat_data.append({
        "commodity": comm, "total": total,
        "Resource (%)": round(n_res / total * 100, 1),
        "Reserve (%)": round(n_rev / total * 100, 1),
        "Resource (n)": int(n_res), "Reserve (n)": int(n_rev),
    })

hdf = pd.DataFrame(heat_data).sort_values("total", ascending=False)
matrix = hdf[["Resource (%)", "Reserve (%)"]].values

im = ax4.imshow(matrix.T, aspect="auto", cmap="RdYlGn", vmin=0, vmax=100)
ax4.set_xticks(range(len(hdf)))
ax4.set_xticklabels(hdf["commodity"], rotation=45, ha="right", fontsize=8)
ax4.set_yticks([0, 1]); ax4.set_yticklabels(["Resource", "Reserve"], fontsize=9)

for i in range(len(hdf)):
    for j, col_type in enumerate(["Resource", "Reserve"]):
        pct = matrix[i, j]
        n = hdf.iloc[i][f"{col_type} (n)"]
        total = hdf.iloc[i]["total"]
        color = "white" if pct < 40 else "black"
        ax4.text(i, j, f"{pct:.0f}%\n({n}/{total})",
                 ha="center", va="center", fontsize=7.5, color=color, fontweight="bold")

cbar = fig.colorbar(im, ax=ax4, orientation="horizontal", pad=0.15, shrink=0.4, aspect=30)
cbar.set_label("% of deposits with data", fontsize=8)
ax4.set_title("Resource vs Reserve Coverage by Commodity (sorted by deposit count)",
              fontweight="bold", pad=10)

out_path = os.path.join(OUT_DIR, "Missing_Data_Analysis.png")
fig.savefig(out_path, dpi=200, bbox_inches="tight")
print(f"Saved to {out_path}")
plt.show()